In [1]:
from data_handler import DataHandlerModule
from model_handler import ModelHandlerModule

config = {
    "seed":                2,
    "data_name":           "ieee_cis",
    "raw_dir":             "./data",  # where CSVs live
    "sample":              10000,     # small sample to verify
    "multi_relation":      True,
    "n_head":              [2, 2],
    "n_head_agg":          8,
    "feat_drop":           0,
    "attn_drop":           0,
    "train_ratio":         0.1,
    "test_ratio":          0.67,
    "emb_size":            [64, 64],
    "lr":                  0.01,
    "weight_decay":        0.001,
    "epochs":              50,        # small for quick check
    "valid_epochs":        10,
    "batch_size":          1024,
    "patience":            20,
    "cuda_id":             0,
    "save_dir":            "./results/ieee_cis",
    "apply_gan":           False,
    "apply_graph_gan":     False,
    "apply_smote":         False,    
    "apply_graph_smote":   False,
    "use_embedding_smote": False, 
}

In [2]:
data_handler  = DataHandlerModule(config)
model_handler = ModelHandlerModule(config, data_handler)
model_handler.train()

drag_model = model_handler.model
graph      = data_handler.dataset['graph']

Loading and preprocessing the dataset ieee_cis...
[IEEE-CIS] Loading from ./data ...... (GAN: False, GraphGAN: False, SMOTE: False, GraphSMOTE: False)
  Sampled:   500000 rows  (fraud=17495, legit=482505)
  Nodes:     500,000
  Etypes:    ['addr_link', 'card_link', 'time_link']
  Features:  369
  Labels:    [482505  17495]
Finished data loading and preprocessing!

 ********************  Train the DRAG  ********************
Epoch: 1 (Best: 0), loss: 0.0008354252139019353, time: 0.2457728385925293s
Epoch: 2 (Best: 0), loss: 0.0007106771705689553, time: 0.10279130935668945s
Epoch: 3 (Best: 0), loss: 0.0006761063297200843, time: 0.10994243621826172s
Epoch: 4 (Best: 0), loss: 0.0006491740699449476, time: 0.10769867897033691s
Epoch: 5 (Best: 0), loss: 0.0006314257807729366, time: 0.11886191368103027s
Epoch: 6 (Best: 0), loss: 0.0006100608329816339, time: 0.10851693153381348s
Epoch: 7 (Best: 0), loss: 0.0005982116491846071, time: 0.1505298614501953s
Epoch: 8 (Best: 0), loss: 0.000602106347984

/mnt/c/Users/Goofy/Documents/Projects/IFT-6759-AP-DRAG_Augmentation/model_handler.py:208: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(self

Test performance: - Epoch_Best: 49	- F1: 0.2531	- Recall: 0.6608	- Precision: 0.1566	- Accuracy: 0.8636	- AUC-ROC: 0.8503	- F1-macro: 0.5890	- Recall-macro: 0.7659	- AP: 0.5713	



In [1]:
from data_handler import DataHandlerModule
from model_handler import ModelHandlerModule

config = {
    "seed":                2,
    "data_name":           "ieee_cis",
    "raw_dir":             "./data",  # where CSVs live
    "sample":              10000,     # small sample to verify
    "multi_relation":      True,
    "n_head":              [2, 2],
    "n_head_agg":          8,
    "feat_drop":           0,
    "attn_drop":           0,
    "train_ratio":         0.1,
    "test_ratio":          0.67,
    "emb_size":            [64, 64],
    "lr":                  0.01,
    "weight_decay":        0.001,
    "epochs":              50,        # small for quick check
    "valid_epochs":        10,
    "batch_size":          1024,
    "patience":            20,
    "cuda_id":             0,
    "save_dir":            "./results/ieee_cis",
    "apply_gan":           False,
    "apply_graph_gan":     False,
    "apply_smote":         False,    
    "apply_graph_smote":   True,
    "use_embedding_smote": True, 
}

In [2]:
import torch
from utils import extract_embeddings

In [ ]:
data_handler  = DataHandlerModule(config)
model_handler = ModelHandlerModule(config, data_handler)
if (config["use_embedding_smote"],  False):
    model_handler.pretrain_embeddings(epochs=5)

    # STEP 5: Extract embeddings
    embeddings = extract_embeddings(model_handler, data_handler)

    # STEP 6: Inject embeddings
    data_handler.embeddings = embeddings

    # STEP 7: Rebuild dataset with GraphSMOTE
    print("Rebuilding dataset with embedding GraphSMOTE...")
    data_handler = DataHandlerModule(config, embeddings=embeddings)

    # Rebuild model
    model_handler = ModelHandlerModule(config, data_handler)
    
model_handler.train()

drag_model = model_handler.model
graph      = data_handler.dataset['graph']

Loading and preprocessing the dataset ieee_cis...
[IEEE-CIS] Loading from ./data ...... (GAN: False, GraphGAN: False, SMOTE: False, GraphSMOTE: True)


In [ ]:
from post_training.contrastive_drag import run_contrastive_pipeline
import copy

# Keep the original untouched
drag_model_original = drag_model

# Each call gets its own independent copy of the baseline weights
drag_model_direct_inference = run_contrastive_pipeline(
    copy.deepcopy(drag_model_original), graph, config,
    data_handler=data_handler, run_3a=True, run_3b=False
)


In [ ]:
drag_model_finetuned = run_contrastive_pipeline(
    copy.deepcopy(drag_model_original), graph, config,
    data_handler=data_handler, run_3a=False, run_3b=True
)